# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ali0369/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [15]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("Token loaded:", HF_TOKEN is not None)

Token loaded: True


In [2]:
from datasets import load_dataset
ds = load_dataset("FlyRank/internship-warehouse", "fact_content_daily_performance", streaming=True, split="train")



README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

In [6]:
print(ds)

IterableDataset({
    features: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events'],
    num_shards: 18
})


In [7]:
%pip -q install duckdb

In [8]:
import duckdb

con = duckdb.connect()

In [16]:
con.execute(f"""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    )
""")

In [17]:
REL = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')"

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of analysis: One row represents the daily performance of one content item for one client on one reporting date.**

**Time window: I am using the March 2026 partition of fact_content_daily_performance, covering March 1, 2026 through March 31, 2026**.

**Verification: The March partition contains 9,841,378 rows from March 1 through March 31. The grain check returned no duplicate client-content-date combinations, supporting the stated unit of analysis.**

In [22]:
next(iter(ds))

{'report_date': datetime.date(2025, 1, 27),
 'client_hash_id': 'client_9958f0a7ae1df715',
 'content_hash_id': 'content_3b70a18ea133b2bb',
 'client_has_gsc': True,
 'client_has_ga4': True,
 'gsc_data_available': True,
 'ga4_data_available': False,
 'gsc_impressions': 30,
 'gsc_clicks': 0,
 'gsc_sum_position': 115,
 'gsc_avg_position': 3.8333333333333335,
 'ga4_pageviews': 0,
 'ga4_sessions': 0,
 'ga4_users': 0,
 'ga4_engaged_sessions': 0,
 'ga4_total_engagement_sec': 0,
 'sessions_organic': 0,
 'sessions_direct': 0,
 'sessions_referral': 0,
 'sessions_social': 0,
 'sessions_paid': 0,
 'sessions_ai': 0,
 'ai_chatgpt': 0,
 'ai_perplexity': 0,
 'ai_gemini': 0,
 'ai_copilot': 0,
 'ai_claude': 0,
 'ai_meta': 0,
 'ai_other': 0,
 'scroll_events': 0}

In [18]:
con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM {REL}
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,row_count,first_date,last_date
0,9841378,2026-03-01,2026-03-31


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

1. **Features:** I will initially use up to five performance signals that are available at the time a refresh decision would be made: gsc_impressions, gsc_clicks, gsc_avg_position, ga4_sessions, and ga4_engaged_sessions.

2. **Label / proxy:** The future refresh outcome or priority signal will be treated as the label/proxy rather than a feature. Any field derived from the future outcome will not be used as an input feature.

3. **Context:** client_hash_id, content_hash_id, and report_date are context fields. They identify the client, content item, and reporting date and are used for grouping, joining, splitting, and understanding the data, not as model features.

4. **Excluded:** Fields that are unavailable at the decision moment, derived from the outcome, or affected by future information will be excluded to avoid leakage. ga4_* measures will also only be used when ga4_data_available IS TRUE.

## 3. Verify it with queries (grain, counts, missing values, windows)



**Grain verification:** I checked whether any combination of reporting date, client, and content appeared more than once. The query returned no duplicate combinations, supporting the stated daily client-content grain.

**Counts and window:** The March 2026 partition contains 9,841,378 rows, covering March 1, 2026 through March 31, 2026.

In [19]:
## GRAIN VERIFICATION
con.sql(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS n
    FROM {REL}
    GROUP BY
        report_date,
        client_hash_id,
        content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,n


In [ ]:
# COUNT & WINDOW VERIFICATION
con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM {REL}
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,row_count,first_date,last_date
0,9841378,2026-03-01,2026-03-31


In [23]:
# AVAILABILITY & MISSINGNESS
con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS gsc_available_rows,
        COUNT(*) FILTER (
            WHERE ga4_data_available IS TRUE
        ) AS ga4_available_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS NOT TRUE
        ) AS gsc_unavailable_rows,
        COUNT(*) FILTER (
            WHERE ga4_data_available IS NOT TRUE
        ) AS ga4_unavailable_rows
    FROM {REL}
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_available_rows,ga4_available_rows,gsc_unavailable_rows,ga4_unavailable_rows
0,9841378,3611061,413966,6230317,9427412


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Data limits:** The data does not provide equally long histories for every client, so the same calendar period may represent different amounts of historical information across clients. GA4 data is also not available for every row, so zero values cannot automatically be interpreted as zero engagement. Finally, this daily performance data describes observed performance but does not by itself explain why a page performed that way or prove that updating a page will improve its future performance.

In [24]:
con.sql(f"""
    SELECT
        client_hash_id,
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (
            WHERE ga4_data_available IS TRUE
        ) AS ga4_available_rows,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM {REL}
    GROUP BY client_hash_id
    ORDER BY total_rows DESC
    LIMIT 10
""").df()

,client_hash_id,total_rows,ga4_available_rows,first_date,last_date
0,client_625b6439094e23e4,988497,37,2026-03-01,2026-03-31
1,client_3ffa76342f366962,904847,4640,2026-03-01,2026-03-31
2,client_73cda7b4e4f265ea,869640,38268,2026-03-01,2026-03-31
3,client_08a6a72ff48e62c0,851275,0,2026-03-01,2026-03-31
4,client_62f4a7e64f5e0096,756660,0,2026-03-01,2026-03-31
5,client_65de48885f4ef01b,426307,4170,2026-03-01,2026-03-31
6,client_23a62021009f63c4,423613,146493,2026-03-01,2026-03-31
7,client_ba65e80a1116ae41,410409,10523,2026-03-01,2026-03-31
8,client_2b4306c3ed003f01,375906,0,2026-03-01,2026-03-31
9,client_fef1a8f436438636,335379,48001,2026-03-01,2026-03-31


**Observed limitation:** The client-level results show that availability and history are not necessarily uniform across clients. This means comparisons should account for differences in data coverage rather than assuming every client has equivalent history.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.